# ▶ Full reproduction — one notebook

Reproduces the **entire** project end-to-end on Yambda-50M: data → baselines →
SASRec → joint-fusion hybrid (negative result) → late fusion → robustness → tail
analysis, ending in one summary table. SASRec is trained **once** and reused, so
this is faster than running the per-stage notebooks.

**How to run:** *Settings → Accelerator =* **GPU** (T4), *Internet =* **On**, then
**Run All**. Total ≈ 45–55 min on a T4. (Optional: `HF_TOKEN` in *Add-ons → Secrets*.)

Code: https://github.com/ak1232320/nndl_capstone_2026

In [1]:
!pip install -q --no-cache-dir --upgrade "ymrec[baselines] @ git+https://github.com/ak1232320/nndl_capstone_2026.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 85.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"  # 13.8 GB embeddings -> 20 GB working disk
import time, numpy as np, pandas as pd, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
from ymrec.config import TOPK
K = max(TOPK)
EPOCHS = 120          # SASRec / hybrid epochs (lower for a quick smoke run)
results = {}          # model -> NDCG@10, filled as we go
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA: True Tesla T4


## 1 · Data — Global Temporal Split, Listen+, shared vocab

`prep` builds the sparse user×item matrix (baselines); `build_sequences` builds
per-user histories (SASRec); both share the same split + item vocabulary. Audio
embeddings are filtered to that vocab.

In [3]:
from ymrec.data.prep import prepare
from ymrec.data.sequences import build_sequences
from ymrec.data.embeddings import load_aligned_embeddings

p = prepare(size="50m")
data = build_sequences(size="50m", maxlen=200)
content_emb, content_mask, dim = load_aligned_embeddings(data.item_ids)
assert np.array_equal(p.item_ids, data.item_ids)  # baselines & neural models share one vocab
print(f"users={p.n_users:,}  items={p.n_items:,}  eval_users={len(p.eval_user_idx):,}  "
      f"audio_dim={dim}  audio_coverage={content_mask.mean():.3f}")

flat/50m/listens.parquet:   0%|          | 0.00/369M [00:00<?, ?B/s]

embeddings.parquet:   0%|          | 0.00/13.8G [00:00<?, ?B/s]

users=9,208  items=629,373  eval_users=4,549  audio_dim=128  audio_coverage=0.946


## 2 · Baselines — MostPop & ItemKNN

In [4]:
from ymrec.eval.metrics import evaluate_ranking
from ymrec.baselines.item_knn import fit_recommend

def ndcg(recs):
    return evaluate_ranking(recs, p.relevant, n_items=p.n_items, ks=TOPK)["ndcg@10"]

pop = np.asarray(p.train_ui.sum(axis=0)).ravel()
mostpop = np.tile(p.item_ids[np.argsort(-pop)[:K]], (len(p.eval_user_idx), 1))
results["MostPop"] = ndcg(mostpop)
print(f"MostPop          NDCG@10 = {results['MostPop']:.4f}")

for v in ["cosine", "tfidf", "bm25"]:
    t = time.time()
    recs = fit_recommend(p.train_ui, p.eval_user_idx, p.item_ids, K, neighbors=100, variant=v)
    results[f"ItemKNN-{v}"] = ndcg(recs)
    print(f"ItemKNN-{v:<6}   NDCG@10 = {results[f'ItemKNN-{v}']:.4f}   [{time.time()-t:.0f}s]")

MostPop          NDCG@10 = 0.0171


/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.06000208854675293 seconds
  warnings.warn(


ItemKNN-cosine   NDCG@10 = 0.0418   [197s]


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.06114315986633301 seconds
  warnings.warn(


ItemKNN-tfidf    NDCG@10 = 0.0451   [191s]


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.056539058685302734 seconds
  warnings.warn(


ItemKNN-bm25     NDCG@10 = 0.0709   [181s]


## 3 · SASRec (sequence model) — trained once, reused below

In [5]:
from ymrec.models.sasrec import train_and_eval as train_sasrec
t = time.time()
sasrec, sasrec_best = train_sasrec(data, d=64, n_blocks=2, n_heads=1, dropout=0.2,
                                   epochs=EPOCHS, batch_size=128, lr=1e-3, eval_every=20)
results["SASRec"] = sasrec_best["ndcg@10"]
print(f"\nSASRec NDCG@10 = {results['SASRec']:.4f}  [{(time.time()-t)/60:.1f} min]")

epoch  20  loss=0.2097  NDCG@10=0.0288  NDCG@100=0.0407  Recall@100=0.0664
epoch  40  loss=0.0916  NDCG@10=0.0457  NDCG@100=0.0657  Recall@100=0.1069
epoch  60  loss=0.0625  NDCG@10=0.0575  NDCG@100=0.0814  Recall@100=0.1266
epoch  80  loss=0.0531  NDCG@10=0.0668  NDCG@100=0.0932  Recall@100=0.1433
epoch 100  loss=0.0418  NDCG@10=0.0703  NDCG@100=0.0976  Recall@100=0.1513
epoch 120  loss=0.0405  NDCG@10=0.0726  NDCG@100=0.1005  Recall@100=0.1533

SASRec NDCG@10 = 0.0726  [16.9 min]


## 4 · Hybrid, attempt 1 — joint embedding fusion (negative result)

Content folded into the item embedding and trained end-to-end. Expected to
**overfit and lose** to SASRec — motivating the late-fusion design next.

In [ ]:
from ymrec.models.hybrid import train_and_eval as train_hybrid
t = time.time()
_, hyb_best = train_hybrid(data, content_emb, content_mask, d=64, n_blocks=2, n_heads=1,
                           dropout=0.2, epochs=EPOCHS, batch_size=128, lr=1e-3, eval_every=20)
results["Hybrid (joint fusion)"] = hyb_best["ndcg@10"]
print(f"\nHybrid joint NDCG@10 = {results['Hybrid (joint fusion)']:.4f}  "
      f"(vs SASRec {results['SASRec']:.4f})  [{(time.time()-t)/60:.1f} min]")

epoch  20  loss=0.1028  NDCG@10=0.0366  NDCG@100=0.0551  Recall@100=0.0891  alpha=1.404
epoch  40  loss=0.0688  NDCG@10=0.0430  NDCG@100=0.0677  Recall@100=0.1136  alpha=1.363
epoch  60  loss=0.0529  NDCG@10=0.0470  NDCG@100=0.0758  Recall@100=0.1273  alpha=1.299
epoch  80  loss=0.0447  NDCG@10=0.0525  NDCG@100=0.0829  Recall@100=0.1379  alpha=1.252
epoch 100  loss=0.0352  NDCG@10=0.0563  NDCG@100=0.0887  Recall@100=0.1462  alpha=1.215
epoch 120  loss=0.0338  NDCG@10=0.0577  NDCG@100=0.0918  Recall@100=0.1519  alpha=1.188

Hybrid joint NDCG@10 = 0.0577  (vs SASRec 0.0726)  [16.8 min]


## 5 · Hybrid, attempt 2 — late fusion (the winner)

Frozen SASRec + a frozen content score, weight β tuned on a **validation** split
of users, reported on held-out test users. β = 0 recovers SASRec, so it cannot lose.

In [ ]:
from ymrec.models.fusion import tune_fusion
fus = tune_fusion(sasrec, content_emb, data, val_frac=0.5, seed=0)
results["Hybrid (late fusion)"] = fus["test_fused"]["ndcg@10"]
print(f"\nbest beta = {fus['best_beta']}")
print(f"test split: SASRec-only {fus['test_sasrec_only']['ndcg@10']:.4f}  ->  "
      f"fused {fus['test_fused']['ndcg@10']:.4f}")

  beta=0.0   val NDCG@10=0.0723
  beta=0.05  val NDCG@10=0.0746
  beta=0.1   val NDCG@10=0.0762
  beta=0.2   val NDCG@10=0.0791
  beta=0.3   val NDCG@10=0.0799
  beta=0.5   val NDCG@10=0.0800
  beta=0.75  val NDCG@10=0.0798
  beta=1.0   val NDCG@10=0.0794
  beta=1.5   val NDCG@10=0.0770
  beta=2.0   val NDCG@10=0.0752

best beta = 0.5
test split: SASRec-only 0.0729  ->  fused 0.0777


## 6 · Robustness — 5 held-out user splits

In [ ]:
from ymrec.models.fusion import robustness
rob = robustness(sasrec, content_emb, data, seeds=(0, 1, 2, 3, 4))
s = rob["summary"]
print(f"\nSASRec {s['sasrec_mean']:.4f}±{s['sasrec_std']:.4f}  ->  "
      f"Fused {s['fused_mean']:.4f}±{s['fused_std']:.4f}   "
      f"lift {s['lift_mean_%']:+.1f}% ± {s['lift_std_%']:.1f}%")

seed 0: beta=0.5   SASRec=0.0729 Fused=0.0777 lift=+6.6%
seed 1: beta=0.75  SASRec=0.0741 Fused=0.0775 lift=+4.5%
seed 2: beta=0.75  SASRec=0.0758 Fused=0.0771 lift=+1.8%
seed 3: beta=0.5   SASRec=0.0753 Fused=0.0805 lift=+6.9%
seed 4: beta=0.5   SASRec=0.0709 Fused=0.0780 lift=+10.1%

SASRec 0.0738±0.0018  ->  Fused 0.0781±0.0012   lift +6.0% ± 2.8%


## 7 · Tail analysis — where does the audio content help?

In [ ]:
from ymrec.models.fusion import tail_analysis
tail = tail_analysis(sasrec, content_emb, data, beta=fus["best_beta"], thresholds=(5, 20, 100))
pd.DataFrame(tail)

tail<= 5     users=3069  SASRec=0.0126 Fused=0.0121 lift=-3.7%
head> 5      users=4444  SASRec=0.0702 Fused=0.0766 lift=+9.1%
tail<= 20    users=3853  SASRec=0.0280 Fused=0.0287 lift=+2.3%
head> 20     users=4256  SASRec=0.0609 Fused=0.0669 lift=+9.8%
tail<= 100   users=4361  SASRec=0.0486 Fused=0.0517 lift=+6.4%
head> 100    users=3714  SASRec=0.0451 Fused=0.0495 lift=+9.8%


,slice,users,sasrec_ndcg@10,fused_ndcg@10,lift_%
0,tail<= 5,3069,0.0126,0.0121,-3.7
1,head> 5,4444,0.0702,0.0766,9.1
2,tail<= 20,3853,0.0280,0.0287,2.3
3,head> 20,4256,0.0609,0.0669,9.8
4,tail<= 100,4361,0.0486,0.0517,6.4
5,head> 100,3714,0.0451,0.0495,9.8


## 8 · Summary — full model ladder (NDCG@10)

In [ ]:
ladder = pd.Series(results, name="NDCG@10").sort_values().round(4)
print(ladder.to_string())
print(f"\nHeadline: late-fusion hybrid {s['fused_mean']:.4f} vs SASRec {s['sasrec_mean']:.4f} "
      f"= {s['lift_mean_%']:+.1f}% ± {s['lift_std_%']:.1f}% (robust across 5 splits).")
ladder.to_frame()

MostPop                  0.0171
ItemKNN-cosine           0.0418
ItemKNN-tfidf            0.0451
Hybrid (joint fusion)    0.0577
ItemKNN-bm25             0.0709
SASRec                   0.0726
Hybrid (late fusion)     0.0777

Headline: late-fusion hybrid 0.0781 vs SASRec 0.0738 = +6.0% ± 2.8% (robust across 5 splits).


,NDCG@10
MostPop,0.0171
ItemKNN-cosine,0.0418
ItemKNN-tfidf,0.0451
Hybrid (joint fusion),0.0577
ItemKNN-bm25,0.0709
SASRec,0.0726
Hybrid (late fusion),0.0777
